In [2]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv", on_bad_lines="skip")

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [3]:
df = df.sample(2000, random_state=42)

print(df.shape)

(2000, 2)


In [4]:
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())

                                                  review  sentiment
33553  I really liked this Summerslam due to the look...          1
9427   Not many television shows appeal to quite as m...          1
199    The film quickly gets to a major chase scene w...          0
12447  Jane Austen would definitely approve of this o...          1
39489  Expectations were somewhat high for me when I ...          0


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1600
Testing samples: 400


In [6]:
!pip install tiktoken

In [7]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [8]:
sample_tokens = tokenizer.encode(X_train.iloc[0])

print(sample_tokens[:20])
print("Number of tokens:", len(sample_tokens))

[464, 1621, 286, 1717, 2269, 259, 318, 3499, 11, 29550, 3499, 13, 770, 3807, 11, 2158, 11, 3499, 691, 287]
Number of tokens: 393


In [9]:
X_train_tokens = [tokenizer.encode(review) for review in X_train]
X_test_tokens = [tokenizer.encode(review) for review in X_test]

print("Training reviews tokenized:", len(X_train_tokens))
print("Testing reviews tokenized:", len(X_test_tokens))

Training reviews tokenized: 1600
Testing reviews tokenized: 400


In [10]:
MAX_LEN = 200

def pad_or_truncate(sequence, max_len=MAX_LEN):
    if len(sequence) < max_len:
        return sequence + [0] * (max_len - len(sequence))
    else:
        return sequence[:max_len]

X_train_padded = [pad_or_truncate(seq) for seq in X_train_tokens]
X_test_padded = [pad_or_truncate(seq) for seq in X_test_tokens]

print("Length of first training sequence:", len(X_train_padded[0]))
print("Length of first testing sequence:", len(X_test_padded[0]))

Length of first training sequence: 200
Length of first testing sequence: 200


In [11]:
import torch

X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)
print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)

X_train shape: torch.Size([1600, 200])
X_test shape: torch.Size([400, 200])
y_train shape: torch.Size([1600])
y_test shape: torch.Size([400])


In [12]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super(SentimentRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.rnn(x)
        x = self.fc(hidden[-1])
        return x

In [13]:
vocab_size = tokenizer.n_vocab

model = SentimentRNN(vocab_size)

print(model)

SentimentRNN(
  (embedding): Embedding(50257, 64)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function and optimizer ready!")

Loss function and optimizer ready!


In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

print("Training batches:", len(train_loader))

Training batches: 50


In [16]:
for epoch in range(5):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/5, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.7072
Epoch 2/5, Loss: 0.6757
Epoch 3/5, Loss: 0.6541
Epoch 4/5, Loss: 0.6259
Epoch 5/5, Loss: 0.5874


In [17]:
model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test_tensor).float().mean()

print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

Test Accuracy: 50.25%
